### create antibiotics data (with SUB_ICB_code)

In [26]:
import pandas as pd
import re

# loading data 
df_cluster = pd.read_csv('antibiotics_clustered_results.csv')

# regualr expression
def extract_ods(name):
    # 正则表达式解释：
    # - : 匹配连字符和空格
    # ([A-Z0-9]{3,5}) : 捕捉 3 到 5 位的字母或数字（ODS Code 的标准长度）
    # \s*$ : 匹配末尾可能存在的空格并结束
    match = re.search(r'- ([A-Z0-9]{3,5})\s*$', str(name))
    if match:
        return match.group(1)
    return None

# 执行提取
df_cluster['SUB_ICB_Code'] = df_cluster['Area Name'].apply(extract_ods)

# 查看提取结果
print(df_cluster[['Area Name', 'SUB_ICB_Code']].head())

                                   Area Name SUB_ICB_Code
0                 South Yorkshire ICB - 02P           02P
1             Mid and South Essex ICB - 99E           99E
2  Nottingham and Nottinghamshire ICB - 02Q           02Q
3    Lancashire and South Cumbria ICB - 00Q           00Q
4    Lancashire and South Cumbria ICB - 00R           00R


In [27]:
df_cluster.to_csv('antibiotics_with_SUB_ICB_codes.csv', index=False)

### fetch drivers data

#### fetch GPs density, elderly, children propotion

In [28]:
df = pd.read_csv('General_Practice_Level_Detailed.csv')

base_cols = ['SUB_ICB_CODE', 'SUB_ICB_NAME', 'TOTAL_PATIENTS', 'TOTAL_GP_FTE']

elderly_cols = [
    'MALE_PATIENTS_65TO74', 'MALE_PATIENTS_75TO84', 'MALE_PATIENTS_85PLUS',
    'FEMALE_PATIENTS_65TO74', 'FEMALE_PATIENTS_75TO84', 'FEMALE_PATIENTS_85PLUS'
]

children_cols = [
    'MALE_PATIENTS_0TO4', 'MALE_PATIENTS_5TO14',
    'FEMALE_PATIENTS_0TO4', 'FEMALE_PATIENTS_5TO14'
]

final_cols = base_cols + elderly_cols + children_cols

df_subset = df[final_cols].copy()
print(df_subset.head())

  SUB_ICB_CODE                                       SUB_ICB_NAME  \
0          16C  NHS North East and North Cumbria ICB - 16C Tee...   
1          16C  NHS North East and North Cumbria ICB - 16C Tee...   
2          16C  NHS North East and North Cumbria ICB - 16C Tee...   
3          16C  NHS North East and North Cumbria ICB - 16C Tee...   
4          16C  NHS North East and North Cumbria ICB - 16C Tee...   

   TOTAL_PATIENTS  TOTAL_GP_FTE  MALE_PATIENTS_65TO74  MALE_PATIENTS_75TO84  \
0            3880      4.560000                   221                   157   
1           18766     10.266667                  1089                   660   
2           11284      3.773333                   593                   351   
3            7784      6.506667                   533                   457   
4           14729      8.220000                   792                   458   

   MALE_PATIENTS_85PLUS  FEMALE_PATIENTS_65TO74  FEMALE_PATIENTS_75TO84  \
0                    56            

In [29]:
df_agg = df_subset.groupby(['SUB_ICB_CODE', 'SUB_ICB_NAME']).sum().reset_index()

#driver 1: GP density
df_agg['Driver_GP_Density'] = (df_agg['TOTAL_GP_FTE']/df_agg['TOTAL_PATIENTS']) * 1000
#driver 2: elderly porpotion (>65 %)
elderly_sum = df_agg[elderly_cols].sum(axis=1)
df_agg['Driver_Elderly_Pct'] = elderly_sum / df_agg['TOTAL_PATIENTS']
#driver 3: children porpotion (<14 %)
children_sum = df_agg[children_cols].sum(axis=1)
df_agg['Driver_chinldren_Pct'] = children_sum / df_agg['TOTAL_PATIENTS']

df_drivers = df_agg[['SUB_ICB_CODE','Driver_GP_Density','Driver_Elderly_Pct','Driver_chinldren_Pct']]


In [30]:
print(df_drivers.head())

  SUB_ICB_CODE  Driver_GP_Density  Driver_Elderly_Pct  Driver_chinldren_Pct
0          00L           0.744611            0.263396              0.141119
1          00N           0.605492            0.208000              0.156159
2          00P           0.632108            0.201135              0.153721
3          00Q           0.531166            0.144929              0.195152
4          00R           0.616012            0.206467              0.155744


### merge antibiotic data with drivers

#### antibiotic + GP density + elderly% + chiledren%

In [31]:
df_final = pd.merge(
    df_cluster,
    df_drivers,
    left_on = 'SUB_ICB_Code',
    right_on = 'SUB_ICB_CODE',
    how = 'inner' 
)
print(df_final.head())

   Area Code                                  Area Name  Qty_Mean_Current  \
0  E38000006                 South Yorkshire ICB - 02P         126.375080   
1  E38000007             Mid and South Essex ICB - 99E         115.499283   
2  E38000008  Nottingham and Nottinghamshire ICB - 02Q         133.863192   
3  E38000014    Lancashire and South Cumbria ICB - 00Q         119.965815   
4  E38000015    Lancashire and South Cumbria ICB - 00R         119.021610   

   Quality_Mean_Current  Seasonality  Quantity_Trend  Cluster       PC1  \
0                5.2700     1.156840       -7.183323        0 -1.158366   
1                7.6750     1.267248      -10.736861        1  1.156192   
2                6.8825     1.153237      -10.075656        0 -1.148037   
3                5.7900     1.168158      -12.869691        2 -1.687177   
4                8.5850     1.134243      -12.737459        2 -1.174961   

        PC2 SUB_ICB_Code SUB_ICB_CODE  Driver_GP_Density  Driver_Elderly_Pct  \
0  1.1

In [32]:
df_final.shape

(106, 14)

In [33]:
print(f'total sub-icb:{len(df_final)}')

total sub-icb:106


#### +COPD

In [34]:
df_copd = pd.read_csv('253_COPD_QOF_prevalence.csv')
# fillter timepireod: 2024/25 ; ICB sub-location
df_copd_sub = df_copd[
    (df_copd['Time period'] == '2024/25') & 
    (df_copd['Area Type'] == 'ICB sub-locations')
].copy()
#extract sub icb code
df_copd_sub['SUB_ICB_Code'] = df_copd_sub['Area Name'].apply(extract_ods)
df_copd_final = df_copd_sub[['SUB_ICB_Code', 'Parent Code', 'Value']].rename(
    columns={'Value': 'Driver_COPD_Prevalence', 'Parent Code': 'ICB_Link'}
)
print(df_copd_final.head())

     SUB_ICB_Code   ICB_Link  Driver_COPD_Prevalence
2223          02P  E54000061                 3.39955
2224          99E  E54000026                 1.74082
2225          02Q  E54000060                 2.99460
2226          00Q  E54000048                 2.03847
2227          00R  E54000048                 3.90359


#### +ethnicity

In [35]:
df_eth = pd.read_excel('Ethnic_group_by_religion.xlsx', skiprows=8)
df_eth.columns = ['Name', 'Code', 'Total', 'Asian', 'Black', 'Mixed', 'White', 'Other']
# extract sub icb code
df_eth['SUB_ICB_Code'] = df_eth['Name'].apply(extract_ods)
# ethnic porpotion
df_eth['Total'] = pd.to_numeric(df_eth['Total'], errors='coerce')
df_eth['White'] = pd.to_numeric(df_eth['White'], errors='coerce')
df_eth['Driver_Minority_Pct'] = ((df_eth['Total'] - df_eth['White']) / df_eth['Total']) * 100
df_eth_final = df_eth[['SUB_ICB_Code', 'Driver_Minority_Pct']]
print(df_eth_final.head())

  SUB_ICB_Code  Driver_Minority_Pct
0          92G             9.301357
1        M1J4Y            27.907696
2          15E            46.054875
3        D2P2L            31.365817
4          15C            12.678079


#### IMD

In [36]:
df_imd = pd.read_csv('94240_Deprivation score (IMD 2025).csv')
# fillter 
df_imd_clean = df_imd[df_imd['Area Type'] == 'ICBs'].copy()
df_imd_final = df_imd_clean[['Area Code', 'Value']].rename(
    columns={'Area Code': 'ICB_Link', 'Value': 'Driver_IMD_Score'}
)
print(df_imd_final.head())

    ICB_Link  Driver_IMD_Score
1  E54000008            25.531
2  E54000010            20.392
3  E54000011            20.099
4  E54000013            22.239
5  E54000015            19.793


#### mergen data

In [40]:
df_final = pd.merge(df_cluster, df_drivers, left_on='SUB_ICB_Code', right_on='SUB_ICB_CODE', how='inner')
df_final = pd.merge(df_final,df_copd_final,on = 'SUB_ICB_Code',how= 'left')
df_final = pd.merge(df_final,df_eth_final,on = 'SUB_ICB_Code', how='left')
df_final = pd.merge(df_final,df_imd_final, on = 'ICB_Link',how = 'left')
print(df_final.head())

   Area Code                                  Area Name  Qty_Mean_Current  \
0  E38000006                 South Yorkshire ICB - 02P         126.375080   
1  E38000007             Mid and South Essex ICB - 99E         115.499283   
2  E38000008  Nottingham and Nottinghamshire ICB - 02Q         133.863192   
3  E38000014    Lancashire and South Cumbria ICB - 00Q         119.965815   
4  E38000015    Lancashire and South Cumbria ICB - 00R         119.021610   

   Quality_Mean_Current  Seasonality  Quantity_Trend  Cluster       PC1  \
0                5.2700     1.156840       -7.183323        0 -1.158366   
1                7.6750     1.267248      -10.736861        1  1.156192   
2                6.8825     1.153237      -10.075656        0 -1.148037   
3                5.7900     1.168158      -12.869691        2 -1.687177   
4                8.5850     1.134243      -12.737459        2 -1.174961   

        PC2 SUB_ICB_Code SUB_ICB_CODE  Driver_GP_Density  Driver_Elderly_Pct  \
0  1.1

In [42]:
cols_to_drop = ['ICB_Link', 'SUB_ICB_CODE'] 
df_final = df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns])

print(df_final.head())

   Area Code                                  Area Name  Qty_Mean_Current  \
0  E38000006                 South Yorkshire ICB - 02P         126.375080   
1  E38000007             Mid and South Essex ICB - 99E         115.499283   
2  E38000008  Nottingham and Nottinghamshire ICB - 02Q         133.863192   
3  E38000014    Lancashire and South Cumbria ICB - 00Q         119.965815   
4  E38000015    Lancashire and South Cumbria ICB - 00R         119.021610   

   Quality_Mean_Current  Seasonality  Quantity_Trend  Cluster       PC1  \
0                5.2700     1.156840       -7.183323        0 -1.158366   
1                7.6750     1.267248      -10.736861        1  1.156192   
2                6.8825     1.153237      -10.075656        0 -1.148037   
3                5.7900     1.168158      -12.869691        2 -1.687177   
4                8.5850     1.134243      -12.737459        2 -1.174961   

        PC2 SUB_ICB_Code  Driver_GP_Density  Driver_Elderly_Pct  \
0  1.126882        

In [43]:
df_final.shape

(106, 16)

In [45]:
df_final.columns

Index(['Area Code', 'Area Name', 'Qty_Mean_Current', 'Quality_Mean_Current',
       'Seasonality', 'Quantity_Trend', 'Cluster', 'PC1', 'PC2',
       'SUB_ICB_Code', 'Driver_GP_Density', 'Driver_Elderly_Pct',
       'Driver_chinldren_Pct', 'Driver_COPD_Prevalence', 'Driver_Minority_Pct',
       'Driver_IMD_Score'],
      dtype='object')

In [46]:
df_final.to_csv('antibiotic_drivers_final.csv', index=False)